Here we create dataset first.
We convert pdf into images for processing. we use `pdf2image` module to do this work. after that we apply processing to these images.

# Extracting Buendia_Instruccion into images

In [1]:
from pdf2image import convert_from_path
import os

pdf_path = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\books\Buendia - Instruccion.pdf"
save_folder = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\Buendia_Instruccion"  # Your folder to save images
poppler_path = r"C:\Users\WC\Downloads\Release-25.12.0-0\poppler-25.12.0\Library\bin"

os.makedirs(save_folder, exist_ok=True)

# Get total pages first
from pdf2image.pdf2image import pdfinfo_from_path
info = pdfinfo_from_path(pdf_path, poppler_path=poppler_path)
total_pages = info["Pages"]

for page_number in range(1, total_pages + 1):
    pages = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path,
                              first_page=page_number, last_page=page_number)
    save_path = os.path.join(save_folder, f'page_{page_number}.jpg')
    pages[0].save(save_path, 'JPEG')
    print(f"Saved page {page_number}/{total_pages}")

Saved page 1/33
Saved page 2/33
Saved page 3/33
Saved page 4/33
Saved page 5/33
Saved page 6/33
Saved page 7/33
Saved page 8/33
Saved page 9/33
Saved page 10/33
Saved page 11/33
Saved page 12/33
Saved page 13/33
Saved page 14/33
Saved page 15/33
Saved page 16/33
Saved page 17/33
Saved page 18/33
Saved page 19/33
Saved page 20/33
Saved page 21/33
Saved page 22/33
Saved page 23/33
Saved page 24/33
Saved page 25/33
Saved page 26/33
Saved page 27/33
Saved page 28/33
Saved page 29/33
Saved page 30/33
Saved page 31/33
Saved page 32/33
Saved page 33/33


# Cropping double column page of Buendia_Instruccion book

In [14]:
import cv2
import numpy as np
import os
from typing import List, Tuple

def detect_columns(img: np.ndarray, min_width: int = 5, gap_threshold: float = 0.05) -> List[Tuple[int, int]]:
    """Detect column boundaries in an image using vertical projection."""
    if img is None or img.size == 0:
        return []
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Use Otsu's threshold for better adaptivity
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Vertical projection
    vertical_sum = np.sum(thresh, axis=0)
    
    max_sum = np.max(vertical_sum)
    if max_sum == 0:
        return []
    
    threshold = gap_threshold * max_sum
    edges = np.where(vertical_sum > threshold)[0]
    
    if len(edges) == 0:
        return []
    
    # Group consecutive pixels
    columns = []
    start = edges[0]
    prev = edges[0]
    
    for edge in edges[1:]:
        if edge != prev + 1:
            if prev - start >= min_width:
                columns.append((start, prev))
            start = edge
        prev = edge
    
    if prev - start >= min_width:
        columns.append((start, prev))
    
    return columns


# ============ MAIN EXECUTION ============

input_folder = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\Buendia_Instruccion\images_v1"
output_folder = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\Buendia_Instruccion\images_cropped"

# DEBUG: Check if folders exist
print(f"📁 Input folder: {input_folder}")
print(f"   Exists: {os.path.exists(input_folder)}")

if not os.path.exists(input_folder):
    print(f"❌ ERROR: Input folder does not exist!")
    exit(1)

# DEBUG: List files
files = os.listdir(input_folder)
print(f"\n📋 Files found: {len(files)}")
for f in files[:5]:  # Show first 5
    print(f"   - {f}")

# Create output folder
os.makedirs(output_folder, exist_ok=True)
print(f"\n✅ Output folder: {output_folder}")

# Process images
total_crops = 0
processed_count = 0

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        print(f"⏭️  Skipping (not image): {filename}")
        continue
    
    img_path = os.path.join(input_folder, filename)
    print(f"\n🔄 Processing: {filename}")
    
    try:
        img = cv2.imread(img_path)
        
        if img is None:
            print(f"   ❌ Failed to read image (file corrupted or invalid format)")
            continue
        
        print(f"   ✅ Image loaded: {img.shape}")
        
        columns = detect_columns(img, min_width=5, gap_threshold=0.05)
        print(f"   📊 Columns detected: {len(columns)}")
        
        if not columns:
            print(f"   ⚠️  No columns found in {filename}")
            continue
        
        base_name, ext = os.path.splitext(filename)
        
        for idx, (x1, x2) in enumerate(columns, 1):
            col_img = img[:, x1:x2]
            output_path = os.path.join(output_folder, f"{base_name}_col{idx}{ext}")
            cv2.imwrite(output_path, col_img)
            total_crops += 1
            print(f"   ✅ Saved: {base_name}_col{idx}{ext} (width: {x2-x1}px)")
        
        processed_count += 1
            
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")

print(f"\n{'='*50}")
print(f"✅ DONE!")
print(f"   Images processed: {processed_count}")
print(f"   Total crops saved: {total_crops}")
print(f"   Output folder: {output_folder}")

📁 Input folder: C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\Buendia_Instruccion\images_v1
   Exists: True

📋 Files found: 33
   - page_1.jpg
   - page_10.jpg
   - page_11.jpg
   - page_12.jpg
   - page_13.jpg

✅ Output folder: C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\Buendia_Instruccion\images_cropped

🔄 Processing: page_1.jpg
   ✅ Image loaded: (6000, 10667, 3)
   📊 Columns detected: 13
   ✅ Saved: page_1_col1.jpg (width: 7px)
   ✅ Saved: page_1_col2.jpg (width: 28px)
   ✅ Saved: page_1_col3.jpg (width: 11px)
   ✅ Saved: page_1_col4.jpg (width: 24px)
   ✅ Saved: page_1_col5.jpg (width: 44px)
   ✅ Saved: page_1_col6.jpg (width: 7px)
   ✅ Saved: page_1_col7.jpg (width: 12px)
   ✅ Saved: page_1_col8.jpg (width: 1512px)
   ✅ Saved: page_1_col9.jpg (width: 8px)
   ✅ Saved: page_1_col10.jpg (width: 79px)
   ✅ Saved: page_1_col11.jpg (width: 211px)
   ✅ Saved: page_1_col12.jpg (width: 264

## Extracting pages of PORCONES_23_5_1628 book into images

In [2]:
# Much simpler and faster!
import fitz  # PyMuPDF
import os

pdf_path = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\books\PORCONES_23_5_1628.pdf"
save_folder = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\PORCONES_23_5_1628\images"

os.makedirs(save_folder, exist_ok=True)

print(f"⏳ Converting PDF to images...")

try:
    pdf_document = fitz.open(pdf_path)
    
    for page_num in range(len(pdf_document)):
        # Render page to image
        page = pdf_document[page_num]
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for quality
        
        # Save
        save_path = os.path.join(save_folder, f'page_{page_num + 1:03d}.png')
        pix.save(save_path)
        
        print(f"✅ Page {page_num + 1:03d}/{len(pdf_document)}")
    
    print(f"\n✅ SUCCESS! {len(pdf_document)} pages converted")
    
except Exception as e:
    print(f"❌ ERROR: {e}")

⏳ Converting PDF to images...
✅ Page 001/12
✅ Page 002/12
✅ Page 003/12
✅ Page 004/12
✅ Page 005/12
✅ Page 006/12
✅ Page 007/12
✅ Page 008/12
✅ Page 009/12
✅ Page 010/12
✅ Page 011/12
✅ Page 012/12

✅ SUCCESS! 12 pages converted


## Extracting pages of PORCONES_228.38_1646 book into images

In [ ]:
import fitz  # PyMuPDF
import os
from pathlib import Path

pdf_path = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\books\PORCONES_23_5_1628.pdf"
save_folder = r"C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\PORCONES_23_5_1628\images"

# Create output folder
Path(save_folder).mkdir(parents=True, exist_ok=True)

# Check file exists
if not os.path.exists(pdf_path):
    print(f"❌ PDF not found: {pdf_path}")
    exit(1)

print(f"📄 Opening: {pdf_path}")

try:
    # Open PDF
    pdf_document = fitz.open(pdf_path)
    total_pages = len(pdf_document)
    print(f"📊 Total pages: {total_pages}\n")
    
    # Convert each page
    for page_num in range(total_pages):
        print(f"⏳ Processing page {page_num + 1}/{total_pages}...", end=" ")
        
        # Get page
        page = pdf_document[page_num]
        
        # Render to image (2x zoom for better quality)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        
        # Save
        save_path = os.path.join(save_folder, f'page_{page_num + 1:03d}.png')
        pix.save(save_path)
        
        print(f"✅ Saved")
    
    pdf_document.close()
    
    print(f"\n" + "="*60)
    print(f"✅ SUCCESS!")
    print(f"   Converted: {total_pages} pages")
    print(f"   Output folder: {save_folder}")
    print(f"   Images: page_001.png to page_{total_pages:03d}.png")
    print("="*60)
    
except Exception as e:
    print(f"\n❌ ERROR: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

📄 Opening: C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\books\PORCONES_23_5_1628.pdf
📊 Total pages: 12

⏳ Processing page 1/12... ✅ Saved
⏳ Processing page 2/12... ✅ Saved
⏳ Processing page 3/12... ✅ Saved
⏳ Processing page 4/12... ✅ Saved
⏳ Processing page 5/12... ✅ Saved
⏳ Processing page 6/12... ✅ Saved
⏳ Processing page 7/12... ✅ Saved
⏳ Processing page 8/12... ✅ Saved
⏳ Processing page 9/12... ✅ Saved
⏳ Processing page 10/12... ✅ Saved
⏳ Processing page 11/12... ✅ Saved
⏳ Processing page 12/12... ✅ Saved

✅ SUCCESS!
   Converted: 12 pages
   Output folder: C:\Open Source\GSOC\org\HumanAI\gsoc-2026-renaissance-ocr-test\GSOC_2026_HumanAI_Test\data\raw\PORCONES_23_5_1628\images
   Images: page_001.png to page_012.png
